# cvpr2024-face-anti-spoofing-challenge

https://github.com/sheriff1max/cvpr2024-face-anti-spoofing-challenge/blob/main/test_batch.py

In [1]:
def hour2sec(hour: float) -> float:
    return hour * 60 * 60

In [2]:
PARAMS = {
    'domain': 'figure',  # cutout | figure | mannequin | mask | print | screen
    'description': 'CVPR2024-FAS',
    'direction': 'maximize',

    'n_trials': 500,
    'timeout': hour2sec(7),  # нужно в секундах
}


_domains = ['cutout', 'figure', 'mannequin', 'mask', 'print', 'screen']
PARAMS['exclude_folders'] = [domain for domain in _domains if domain != PARAMS['domain']]
print(f"{PARAMS['exclude_folders'] = }")

PARAMS['description'] += f' domain={PARAMS["domain"]}'
print(f"{PARAMS['description'] = }")

PARAMS['exclude_folders'] = ['cutout', 'mannequin', 'mask', 'print', 'screen']
PARAMS['description'] = 'CVPR2024-FAS domain=figure'


In [3]:
!git clone https://github.com/sheriff1max/cvpr2024-face-anti-spoofing-challenge.git

Cloning into 'cvpr2024-face-anti-spoofing-challenge'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 127 (delta 46), reused 110 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 3.36 MiB | 13.64 MiB/s, done.
Resolving deltas: 100% (46/46), done.


In [4]:
!git clone https://github.com/sheriff1max/fas_aug_attack.git -q

In [5]:
!pip install thop -q
!pip install elasticdeform==0.5.0 -q

  Preparing metadata (setup.py) ... done


In [6]:
import sys


path_model = '/kaggle/working/cvpr2024-face-anti-spoofing-challenge'
if path_model not in sys.path:
    sys.path.insert(1, path_model)

path_attack = '/kaggle/working/fas_aug_attack'
if path_attack not in sys.path:
    sys.path.insert(1, path_attack)

In [7]:
import builtins
import os
import sys
import random
import shutil
import time
import math
import warnings
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.distributed as dist
import torch.optim
import torch.utils.data
import torch.utils.data.distributed
from nets.utils import get_model, load_pretrain, load_resume, ExponentialMovingAverage
from datasets.utils import get_train_dataset, get_val_dataset

import torchvision

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [9]:
checkpoint = torch.load('/kaggle/input/datasets/morph1max/cvpr2024-fas-challenge-weights/full_resnet50.pth', map_location=device)
new_state_dict = {key[7:]: value for key, value in checkpoint['state_dict'].items()}
checkpoint.keys()

dict_keys(['epoch', 'arch', 'state_dict', 'state_dict_ema', 'best_acc1', 'optimizer', 'scaler'])

In [10]:
print(list(checkpoint['state_dict'].keys())[30])
print(list(new_state_dict.keys())[30])

module.layer1.1.conv1.weight
layer1.1.conv1.weight


In [11]:
arch = 'resnet50'
num_classes = 2


net = get_model(arch, num_classes)
net.load_state_dict(new_state_dict)

net = net.to(device)

## Dataset

In [12]:
from src.utils.dataset import AttackDataset

dataset = AttackDataset(
    path='/kaggle/input/datasets/morph1max/spoof-attack-liveness-face',
    exclude_folders=PARAMS['exclude_folders'],
)
print(f'Returns keys = {dataset[0].keys()}')
len(dataset)

Returns keys = dict_keys(['img', 'filename', 'path2file', 'is_real', 'type_attack'])


42

## Model

In [13]:
from src.base import BaseModel
from typing import Any


def get_transform(
    img_size: tuple[int] = (224, 224),
    normalize_mean: list[float] = [0.485, 0.456, 0.406],
    normalize_std: list[float] = [0.229, 0.224, 0.225],
):
    return torchvision.transforms.Compose(
        [
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(mean=normalize_mean, std=normalize_std),
            torchvision.transforms.Resize(img_size),
        ]
    )

to_tensor = get_transform()


class CVPR2024_FAS_Model(BaseModel):
    def __init__(self, model: Any):
        super().__init__(model=model)

    def predict(self, img: Any) -> float:
        self.model.eval()

        with torch.no_grad():
            img = to_tensor(img).unsqueeze(0)
            img = img.to(device)

            _, outputs = self.model(img)

            # [live class, attack class]
            score = torch.softmax(outputs, dim=1).data.cpu().numpy()[0, 0]
            return float(score)



    
model = CVPR2024_FAS_Model(model=net)
pred = model.predict(dataset[0]['img'])
print(pred, type(pred))

/kaggle/working/cvpr2024-face-anti-spoofing-challenge/nets/resnet.py:235: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self.fp16):


7.287968628588715e-08 <class 'float'>


## Transforms

In [14]:
from src import transforms
from src.base import BaseTransform
import inspect

In [ ]:
list_type_transforms = [
    # Advanced transformations
    [
        transforms.MorphologicalTransform,
        None,
    ],

    # Noise transforms
    [
        transforms.PixelDropoutTransform,
        None,
    ],

    # Color transformations
    [
        transforms.BrightnessContrastTransform,
        transforms.HSVTransform,
        transforms.RGBShiftTransform,
        transforms.GammaTransform,
        transforms.SolarizeTransform,
        transforms.PosterizeTransform,
        transforms.EqualizeTransform,
        transforms.InvertTransform,
        transforms.ToGrayTransform,
        transforms.ChannelShuffleTransform,
        transforms.ToSepiaTransform,
        None,
    ],

    # Transformations of blurring and sharpness
    [
        transforms.BlurTransform,
        transforms.GaussianBlurTransform,
        transforms.MedianBlurTransform,
        transforms.MotionBlurTransform,
        transforms.SharpenTransform,
        transforms.EmbossTransform,
        None,
    ],

    # Weather and atmospheric conditions
    [
        transforms.RainTransform,
        transforms.SnowTransform,
        transforms.RainbowTransform,
        transforms.ChromaticAberrationTransform,
        transforms.DefocusTransform,
        transforms.ZoomBlurTransform,
        None,
    ],

    # Geometric transformations
    [
        transforms.PerspectiveTransform,
        transforms.OpticalDistortionTransform,
        transforms.ShiftScaleRotateTransform,
        None,
    ],

    # Transformation of compression and artifacts
    [
        transforms.CompressionTransform,
        transforms.DownscaleTransform,
        None,
    ],

    # Dropout transforms
    [
        transforms.CoarseDropoutTransform,
        transforms.GridDropoutTransform,
        None,
    ],
]


list_type_transforms[:5]

[[src.transforms.transforms.PixelDropoutTransform, None],
 [src.transforms.transforms.BrightnessContrastTransform,
  src.transforms.transforms.HSVTransform,
  src.transforms.transforms.RGBShiftTransform,
  src.transforms.transforms.GammaTransform,
  src.transforms.transforms.EqualizeTransform,
  src.transforms.transforms.ToGrayTransform,
  src.transforms.transforms.ChannelShuffleTransform,
  src.transforms.transforms.ToSepiaTransform,
  src.transforms.transforms.RainbowTransform,
  None],
 [src.transforms.transforms.DownscaleTransform,
  src.transforms.transforms.EmbossTransform,
  None]]

## Pipeline

In [16]:
from src.pipeline import PipelineAttackOptunaDataset, PipelineAttackOptunaImg
from src.utils.logging import LoggerOptuna

In [17]:
logger = LoggerOptuna(
    direction=PARAMS['direction'],
    description=PARAMS['description'],
)

# DATASET
optuna_attack_pipeline_dataset = PipelineAttackOptunaDataset(
    model=model,
    list_type_transforms=list_type_transforms,
    logger=logger,
)

optuna_attack_pipeline_dataset.optimize(
    data=dataset,
    direction=PARAMS['direction'],
    n_trials=PARAMS['n_trials'],
    timeout=PARAMS['timeout'],
    show_progress=True,
    catch=(ValueError, ),
)

# IMG
# optuna_attack_pipeline_img = PipelineAttackOptunaImg(
#     model=model,
#     list_type_transforms=list_type_transforms,
#     logger=logger,
# )

# optuna_attack_pipeline_img.optimize(
#     data=dataset[0]['img'],
#     direction=PARAMS['direction'],
#     n_trials=PARAMS['n_trials'],
#     timeout=PARAMS['timeout'],
#     show_progress=True,
#     catch=(ValueError, ),
# )

  0%|          | 0/500 [00:00<?, ?it/s]

In [18]:
print(os.listdir('/kaggle/working/logs'))

['run_1']


In [19]:
# !zip -FSr run.zip /kaggle/working/logs/run_1

In [20]:
shutil.rmtree('cvpr2024-face-anti-spoofing-challenge')
shutil.rmtree('fas_aug_attack')